# Style.io: train the matcher

This notebook fine-tunes the matcher: a small add-on layer (the **head**) on top of FashionCLIP, trained so a Pinterest
piece lands next to the closet item a person actually wore in its place (the stand-in), and away from the others.

FashionCLIP itself stays frozen in this first stage. That's the cheap, safe start with ~80 training pairs. Unfreezing
its last layers is the next stage, once this one is measured.

**Before you run it**
1. Runtime → Change runtime type → **T4 GPU**.
2. Upload `styleio_data.zip` (from `Style.io_Data/colab/` on the Mac) to Google Drive, in a folder called
   `Style.io` (so the path is `MyDrive/Style.io/styleio_data.zip`).
3. Run the cells top to bottom. **Run the test cell only once**, at the end (see why there).

## 1. Setup

Mounts Google Drive, unzips the data, and imports `baseline.py` from the zip. Reusing its functions means pieces and
closet items are read, embedded and scored exactly the way they are on the Mac (rule 1: one processing path).

In [ ]:
import sys, json, random
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/Style.io")
WORK = Path("/content/styleio")
!unzip -q -o "{DRIVE}/styleio_data.zip" -d "{WORK}"
!nvidia-smi -L

sys.path.insert(0, str(WORK))
DATA = WORK / "Style.io_Data"

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import ImageOps
import matplotlib.pyplot as plt

from baseline import (find_people, load_closet, load_answers, load_image, load_model,
                      match, metrics, random_guessing, ranges, TOP_K)

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Running on", device)

## 2. Settings

Everything you might change is here. Change one thing at a time and write down what it did.

- `CROPS`: `"redrawn"` (the product-shot redraws; they scored best untrained) or `"rough"` (rectangle crops).
- `CLOSET`: who a validation or test piece is ranked against. `"plus-train"` = its own group's closets plus the train
  group's, about 100 items, close to a real closet. That's also what makes validation hard enough to be useful.
- `VIEWS`: extra flipped and re-cropped copies of each image, so the head sees each garment a few ways. Colour is never
  changed, because colour is part of what we match on.
- `TEMPERATURE`: how sharply the loss punishes the right item not being first. Lower is sharper.
- `PATIENCE`: stop after this many passes without the validation score improving (early stopping).

In [ ]:
MODEL = "fashion-clip"     # or "clip"
CROPS = "redrawn"          # or "rough"
CLOSET = "plus-train"      # or "group", "own"

VIEWS = 4                  # extra augmented copies per image
HIDDEN = 128               # size of the head's middle layer
DROPOUT = 0.1
TEMPERATURE = 0.05
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-2        # pulls weights towards zero, a guard against memorising
EPOCHS = 300
PATIENCE = 30
SEED = 0

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 3. Read the data

Each person's Pinterest pieces, closet items and answer key, with their group (train, validation or test) from
`split.json`. The group is fixed per person, so nobody's photos, room or lighting appear in two groups.

In [ ]:
people = []   # (group, person, closet items, answer rows)
for group, person in find_people(DATA):
    crops = person if CROPS == "rough" else person / CROPS
    closet = load_closet(crops)
    answers = load_answers(person, crops, closet)
    for item in closet: item["person"] = person.name
    for row in answers:
        row["person"] = person.name
        row["path"] = crops / "pieces" / row["piece"]
    if answers: people.append((group, person.name, closet, answers))

group_of = {name: group for group, name, _, _ in people}
for g in ["train", "validation", "test"]:
    names = [n for n, gg in group_of.items() if gg == g]
    pieces = sum(len(a) for gg, _, _, a in people if gg == g)
    items = sum(len(c) for gg, _, c, _ in people if gg == g)
    print(f"{g:<11} {len(names):>2} people, {pieces:>3} pieces, {items:>3} closet items")

## 4. Turn every image into numbers, once

FashionCLIP turns each image into 512 numbers (an embedding). Because FashionCLIP is frozen, those numbers never change,
so they're worked out once and saved to Drive. Training then only touches the small head, which is why it's fast.

Each image also gets `VIEWS` augmented copies: mirrored, and re-cropped to 80–100% of its size.

In [ ]:
model, processor = load_model(MODEL, device)

def embed_images(images):
    """The same steps as baseline.embed, for images already open: processor, model, then length 1."""
    vectors = []
    for start in range(0, len(images), 32):
        inputs = processor(images=images[start:start + 32], return_tensors="pt").to(device)
        with torch.no_grad():
            output = model.get_image_features(**inputs)
        if not torch.is_tensor(output): output = output.pooler_output
        vectors.append(output.float().cpu())
    vectors = torch.cat(vectors)
    return vectors / vectors.norm(dim=1, keepdim=True)

def views(image, rng):
    """The original first, then VIEWS copies: maybe mirrored, then cropped to a random 80-100% window."""
    out = [image]
    for _ in range(VIEWS):
        v = ImageOps.mirror(image) if rng.random() < 0.5 else image
        w, h = v.size
        scale = rng.uniform(0.8, 1.0)
        cw, ch = int(w * scale), int(h * scale)
        left, top = rng.randint(0, w - cw), rng.randint(0, h - ch)
        out.append(v.crop((left, top, left + cw, top + ch)))
    return out

paths = sorted({str(item["path"]) for _, _, closet, _ in people for item in closet} |
               {str(row["path"]) for _, _, _, answers in people for row in answers})

cache_file = DRIVE / f"cache-{MODEL}-{CROPS}-views{VIEWS}-seed{SEED}.pt"
cache = torch.load(cache_file) if cache_file.exists() else {}
key = lambda p: str(Path(p).relative_to(DATA))
missing = [p for p in paths if key(p) not in cache]
rng = random.Random(SEED)
for i, p in enumerate(missing):
    cache[key(p)] = embed_images(views(load_image(p), rng))
    if (i + 1) % 50 == 0: print(f"  {i + 1}/{len(missing)}")
if missing:
    DRIVE.mkdir(parents=True, exist_ok=True)
    torch.save(cache, cache_file)
print(f"{len(paths)} images ready, {len(missing)} newly embedded. Each is {tuple(cache[key(paths[0])].shape)} (views x numbers)")

E = lambda p: cache[key(p)]

## 5. The head

A small add-on after FashionCLIP: squeeze the 512 numbers down to `HIDDEN`, back up to 512, and **add** that to the
original. The last layer starts at zero, so before training the head adds nothing and scores exactly like the
untrained baseline. Training can only move it away from there if that helps.

The same head is used for Pinterest pieces and closet items, so both sides stay comparable.

In [ ]:
class Head(nn.Module):
    def __init__(self, size=512, hidden=HIDDEN, dropout=DROPOUT):
        super().__init__()
        self.down = nn.Linear(size, hidden)
        self.up = nn.Linear(hidden, size)
        self.drop = nn.Dropout(dropout)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        x = x + self.up(self.drop(torch.relu(self.down(x))))
        return F.normalize(x, dim=-1)   # back to length 1, so a dot product is still cosine similarity

print(sum(p.numel() for p in Head().parameters()), "numbers to learn")

## 6. Scoring, the same way as baseline.py

For each piece in a group, rank the closet items of the same category from the pool (see `CLOSET`), using the original
image (view 0, no augmentation). Then Recall@1, Recall@3 and MRR, worked out per category and averaged.

In [ ]:
all_items = [item for _, _, closet, _ in people for item in closet]

def pool_people(group, person):
    if CLOSET == "own": return {person}
    keep = {group, "train"} if CLOSET == "plus-train" else {group}
    return {name for name, g in group_of.items() if g in keep}

@torch.no_grad()
def evaluate(head, group):
    """Every piece in the group, ranked. head=None means no head: the untrained baseline."""
    apply = (lambda x: x) if head is None else (lambda x: head(x.to(device)).cpu())
    if head is not None: head.eval()
    item_vectors = apply(torch.stack([E(item["path"])[0] for item in all_items])).numpy()
    results = []
    for g, name, closet, answers in people:
        if g != group: continue
        names = pool_people(g, name)
        pool = [i for i, item in enumerate(all_items) if item["person"] in names]
        piece_vectors = apply(torch.stack([E(row["path"])[0] for row in answers])).numpy()
        for r in match(answers, piece_vectors, [all_items[i] for i in pool], item_vectors[pool]):
            results.append({"split": g, **r})
    return results

def scores(results):
    matched = [r for r in results if r["rank"] is not None]
    return metrics(matched), matched

def show(label, results):
    (r1, r3, mrr), matched = scores(results)
    (lo, hi), chance = ranges(matched), random_guessing(matched)
    print(f"{label}: {len(matched)} pieces")
    for i, (name, value) in enumerate([("Recall@1", r1), (f"Recall@{TOP_K}", r3), ("MRR", mrr)]):
        print(f"  {name:<9} {value:.2f} ({lo[i]:.2f}-{hi[i]:.2f})   random: {chance[i]:.2f}")

show("Validation, untrained", evaluate(None, "validation"))

## 7. Train

Each pass (epoch):
1. Pick a random view of every train piece and every train closet item, and put them through the head.
2. For each piece, score every train closet item of the same category. The loss is high when the right item (the
   stand-in) isn't clearly first. This is contrastive learning: pull the right pair together, push the rest apart.
3. Score validation. If it's the best so far, keep a copy of the head. If it hasn't improved for `PATIENCE` passes, stop.

The kept copy is the one from the best validation pass, not the last one, so memorising the train group late in
training can't sneak into the result.

In [ ]:
train_items = [item for g, _, closet, _ in people if g == "train" for item in closet]
index = {(item["person"], item["name"]): i for i, item in enumerate(train_items)}
train_rows = [row for g, _, _, answers in people if g == "train" for row in answers if row["correct_item"] != "none"]

P = torch.stack([E(row["path"]) for row in train_rows]).to(device)        # pieces x views x 512
C = torch.stack([E(item["path"]) for item in train_items]).to(device)     # items x views x 512
target = torch.tensor([index[(row["person"], row["correct_item"])] for row in train_rows], device=device)
same_category = torch.tensor([[row["category"] == item["category"] for item in train_items] for row in train_rows], device=device)
print(len(train_rows), "train pairs,", len(train_items), "train closet items")

head = Head().to(device)
optimizer = torch.optim.AdamW(head.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

history, best, best_state, best_epoch, waited = [], -1.0, None, 0, 0
for epoch in range(1, EPOCHS + 1):
    head.train()
    pick_p = torch.randint(0, VIEWS + 1, (len(P),), device=device)
    pick_c = torch.randint(0, VIEWS + 1, (len(C),), device=device)
    pieces = head(P[torch.arange(len(P)), pick_p])
    items = head(C[torch.arange(len(C)), pick_c])
    logits = pieces @ items.T / TEMPERATURE
    logits = logits.masked_fill(~same_category, float("-inf"))   # only same-category items compete
    loss = F.cross_entropy(logits, target)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

    (r1, r3, mrr), _ = scores(evaluate(head, "validation"))
    history.append((loss.item(), mrr, r1))
    if mrr > best:
        best, best_epoch, waited = mrr, epoch, 0
        best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}
    else:
        waited += 1
    if epoch % 10 == 0 or epoch == 1:
        print(f"epoch {epoch:>3}  loss {loss.item():.3f}  validation MRR {mrr:.3f}  Recall@1 {r1:.2f}")
    if waited >= PATIENCE:
        print(f"Stopped at epoch {epoch}: no improvement for {PATIENCE} passes.")
        break

head.load_state_dict(best_state)
print(f"Kept the head from epoch {best_epoch} (validation MRR {best:.3f}).")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot([h[0] for h in history]); ax[0].set_title("Train loss"); ax[0].set_xlabel("epoch")
ax[1].plot([h[1] for h in history], label="MRR"); ax[1].plot([h[2] for h in history], label="Recall@1")
ax[1].axvline(best_epoch - 1, color="grey", ls="--"); ax[1].set_title("Validation"); ax[1].legend(); ax[1].set_xlabel("epoch")
plt.show()

**Reading the curves.** Train loss should fall. Validation should rise, then level off. If train loss keeps falling
while validation drops, the head is memorising the train group: that's overfitting, and early stopping is what cuts it
off. With only ~10 validation pieces the validation line is jumpy; that's expected at this size.

## 8. Test, once

This scores the test group (model_001 to model_004: the four people under "test" in `split.json`) with and without the head.

**Run it once per version, and don't change settings because of what it shows.** If you tweak until test looks good,
the test group has quietly become training data and the score stops meaning anything. Tweak using validation only.

In [ ]:
before, after = evaluate(None, "test"), evaluate(head, "test")
show("Test, untrained baseline", before)
show("Test, with the trained head", after)

changed = [(b["piece"], b["rank"], a["rank"], a["top_guess"]) for b, a in zip(before, after)
           if b["rank"] is not None and b["rank"] != a["rank"]]
print(f"\n{len(changed)} pieces moved:")
for piece, rb, ra, guess in sorted(changed, key=lambda c: c[2] - c[1]):
    print(f"  {'better' if ra < rb else 'worse ':<6} {piece:<40} rank {rb} -> {ra}   now first: {guess}")

## 9. Save

The trained head and a record of this run go to Drive. The head is tiny (a few hundred KB). To use it later, load
FashionCLIP as usual, then put its 512 numbers through the head.

In [ ]:
(before_scores, _), (after_scores, _) = scores(before), scores(after)
name = f"head-{MODEL}-{CROPS}-seed{SEED}"
torch.save(head.state_dict(), DRIVE / f"{name}.pt")
record = {
    "settings": {k: globals()[k] for k in ["MODEL", "CROPS", "CLOSET", "VIEWS", "HIDDEN", "DROPOUT", "TEMPERATURE",
                                            "LEARNING_RATE", "WEIGHT_DECAY", "EPOCHS", "PATIENCE", "SEED"]},
    "best_epoch": best_epoch,
    "validation_mrr": best,
    "test_untrained": dict(zip(["recall@1", f"recall@{TOP_K}", "mrr"], map(float, before_scores))),
    "test_trained": dict(zip(["recall@1", f"recall@{TOP_K}", "mrr"], map(float, after_scores))),
}
(DRIVE / f"{name}.json").write_text(json.dumps(record, indent=2))
print(json.dumps(record, indent=2))